[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/frame_fields/frame_fields.ipynb)

# Smooth Frame Fields

This example interpolates 3D rotations attached to the vertices of a triangulated surface into a continuous field of orientations across its faces. Blending corner rotations linearly traces a chord inside the unit sphere of rotations so that both the orientation and the chord length vary across each triangle until radial normalization restores a rigid turn. Following that local construction across whole surfaces produces a planar disk whose frames wrap a chosen director once around the sphere of directions, known in magnetism as a skyrmion, as well as smooth orientation fields guided across a curved saddle by prescribed boundary frames.

Each orientation is modelled as an even rotor, and affine blending with the simplex gradients `1 + (gradients & (points[:, None] - corners))` produces an even chord element inside the unit rotor sphere. Pulling its differential back through the chord reverse, `2 * chord_rotors.reverse() * rotor_differential`, splits the variation cleanly by grade: the bivector grade assembles edge transition planes against antisymmetric edge 1-forms as in [Whitney forms](../whitney_forms/whitney_forms.ipynb), while the scalar grade pairs edge chord sagittas with symmetric edge differentials. Across a triangulated disk, commuting the turning rates along two sides of each triangle measures the solid angle swept out by `rotors >> vertical`, and on a curved saddle surface one linear cotangent solve in the even subalgebra extends boundary rotors across the interior before unit-constrained relaxation smooths interior twist directly on the rotor sphere.

In unit-quaternion and finite-element exterior-calculus notation, blending vertex quaternions $q_i \in \mathbb{H}$ with barycentric coordinates $\lambda_i$ gives $q = \sum_i \lambda_i q_i$, whose Maurer–Cartan pullback decomposes as $2\,\bar q\,dq = d|q|^2 + \sum_{(i,j)} 2\operatorname{Im}(\bar q_i q_j)\,(\lambda_i\,d\lambda_j - \lambda_j\,d\lambda_i)$ while the projected director $\mathbf{n} = q\,\mathbf{e}_3\,\bar q$ carries skyrmion winding $Q = \frac{1}{4\pi}\int \mathbf{n}\cdot(\partial_x\mathbf{n}\times\partial_y\mathbf{n})\,dx\,dy$ and unit-constrained smoothing solves $(Lq)_i = \langle q_i, (Lq)_i\rangle\,q_i$ on $S^3$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import Algebra, NumpyContext, stack
from numga.algebras import VGA3D as ga
from numga.sparse import SparseExtensor, spdiag
from examples.util import shapes
from numga.topology import Topology
from examples.surfaces.frame_fields import render, scenarios
from examples.util.animation import save_figure, show_states

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                  # Scalar+
Plane = ga.gatype.vector()                                                   # Vector+
Bivector = ga.gatype.bivector()                                              # Bivector+
Even = ga.gatype.even()                                                      # Even+
Rotor = ga.gatype.rotor()                                                    # Rotor+
Point = ga.gatype.antivector()                                               # Antivector-
Patch = ga.gatype.antibivector()                                             # Antibivector-
EvenMap = ga.gatype((Even, Point))
RateMap = ga.gatype((Bivector, Point))
StretchMap = ga.gatype((Scalar, Point))
Upright = ga.gatype.vector().derive.within(Algebra("x+y+"))                  # Vector+
triangle = Topology(3, [[0, 1, 2]])

np.set_printoptions(precision=4, suppress=True)

## Turning and chord stretch on a single triangle

Moving across a triangle changes both the orientation and the length of the linearly blended rotor chord. Pulling `rotor_differential` back through `2 * chord_rotors.reverse()` separates those two effects by grade: grade two sums each edge's transition bivector against its antisymmetric Whitney 1-form, while grade zero sums each edge's chord sagitta against the symmetric edge differential.

In [ ]:
corners, points, fine_topology, fine_vertices, corner_rotors = scenarios.single_triangle(
    scenarios.n_triangle_subdivisions, scenarios.n_fine_levels,
)

In [ ]:
def triangle_darboux(
    corners: Point, points: Point, corner_rotors: Rotor,
) -> tuple[Even, Rotor, EvenMap, RateMap, StretchMap, RateMap, Plane]:
    """Linearly blended corner rotors (`[3] Rotor+`) across sample points (`[...] Point-`) decompose their Darboux derivative into antisymmetric edge Whitney 1-forms in grade two and symmetric Regge differentials in grade zero."""
    sides = (triangle.boundary[2, 1] * (triangle.d[0] * corners.field())).batch()[:, 0]  # [3] Point-
    patch = (sides[0] & sides[1]) / 2                                        # [] Patch-
    gradients = (sides | patch.inverse()) / 2                                # [3] Plane+
    corner_weights = 1 + (gradients & (points[:, None] - corners))           # [..., 3] Scalar+

    # Linear blending cuts inside the unit rotor sphere; normalization restores a rigid turn.
    chord_rotors = (corner_rotors * corner_weights).sum(axis=-1)             # [...] Even+
    unit_rotors = chord_rotors.normalized()                                  # [...] Rotor+
    rotor_differential = (corner_rotors * (gradients & Point)).sum(axis=0)   # [] Even+ <- Point-
    darboux = 2 * chord_rotors.reverse() * rotor_differential                # [...] Even+ <- Point-

    # Reversing an edge flips its transition's turning plane and preserves its scalar sagitta.
    tails, heads = triangle.corners[2, 1]
    transitions = corner_rotors[tails].reverse() * corner_rotors[heads]      # [3] Rotor+
    edge_bivectors = 2 * transitions.select[2]                               # [3] Bivector+
    edge_sagittas = 2 * (transitions.select[0] - 1)                          # [3] Scalar+
    whitney_forms = corner_weights[:, tails] * gradients[heads] - corner_weights[:, heads] * gradients[tails]    # [..., 3] Plane+
    symmetric_forms = corner_weights[:, tails] * gradients[heads] + corner_weights[:, heads] * gradients[tails]  # [..., 3] Plane+
    whitney_rates = (edge_bivectors * (whitney_forms & Point)).sum(axis=-1)  # [...] Bivector+ <- Point-
    chord_stretch = (edge_sagittas * (symmetric_forms & Point)).sum(axis=-1) # [...] Scalar+ <- Point-
    angular_velocity = whitney_rates / chord_rotors.scalar_norm_squared()    # [...] Bivector+ <- Point-
    return chord_rotors, unit_rotors, darboux, whitney_rates, chord_stretch, angular_velocity, whitney_forms


_, unit_rotors, darboux, whitney_rates, chord_stretch, _, whitney_forms = triangle_darboux(
    corners, points, corner_rotors,
)
fine_chords, _, _, _, _, _, _ = triangle_darboux(corners, fine_vertices.batch(), corner_rotors)
fine_norms_squared = fine_chords.scalar_norm_squared().field()               # Scalar+[V_fine]

save_figure(
    render.darboux_figure(
        corners, points, fine_topology, fine_vertices, fine_norms_squared,
        corner_rotors, unit_rotors, whitney_rates,
    ),
    "frame_fields_darboux",
);

Four coloured sectors on each rigid tile show the normalized frame orientation at the three corners and across the interior. In the middle panel, coral-to-gold arcs turn inside the bivector planes returned by the Whitney rate map along the blue probe step, while the right panel lifts the squared chord norm as a blue relief bowl pinned to unit height at the gold corner markers.

In [ ]:
print("Minimum squared chord norm across the triangle:", fine_norms_squared.batch().to_array().min())

## Wrapping the director sphere on a planar disk

Tilting the vertical director radially from upward at the disk centre through the horizontal plane to downward all along the rim wraps the sphere of directions once. Although the full rotor connection across the disk has zero curvature, commuting the frame turning rates along two sides of each triangle reads the solid angle swept out by the projected director `rotors >> vertical`.

In [ ]:
skyrmion_topology, skyrmion_coordinates = shapes.triangular_disk(
    scenarios.disk_radius, scenarios.n_skyrmion_divisions,
)
skyrmion_vertices = mv(Upright, skyrmion_coordinates).dual().field()         # Point-[V]

In [ ]:
def skyrmion_winding(
    topology: Topology, vertices: Point, twist_angle: float,
) -> tuple[Rotor, Point, Scalar, Scalar, Scalar, Scalar]:
    """A radial rotor texture (`Rotor+[V]`) wraps the director sphere across a triangulated disk while its face Darboux rates give the frame Dirichlet energy density (`Scalar+[F]`) and swept solid angles (`Scalar+[F]`)."""
    vertical = mv.z.dual()                                                   # [] Point-
    # Radial position and vertical direction span the turning plane; a half turn sends the rim to the south pole.
    rotors = ((-twist_angle / 2) * vertices.commutator(vertical)).exp()      # Rotor+[V]
    directors = rotors >> vertical                                           # Point-[V]
    scalar_parts = rotors.select[0]                                          # Scalar+[V]
    bivector_norms = rotors.select[2].norm()                                 # Scalar+[V]

    edges = topology.d[0] * vertices                                         # Point-[E]
    sides = topology.boundary[2, 1] * edges                                  # [3] Point-[F]
    patches = (sides[0] & sides[1]) / 2                                      # Patch-[F]
    areas = patches.norm()                                                   # Scalar+[F]
    gradients = (sides | patches.inverse()) / 2                              # [3] Plane+[F]

    face_rotors = topology.incidence[2, 0] * rotors                          # [3] Rotor+[F]
    face_chords = face_rotors.sum(axis=0) / 3                                # Even+[F]
    face_differential = (face_rotors * (gradients & Point)).sum(axis=0)      # (Even+ <- Point-)[F]
    face_darboux = 2 * face_chords.reverse() * face_differential             # (Even+ <- Point-)[F]
    face_rates = face_darboux.select[2] / face_chords.scalar_norm_squared()  # (Bivector+ <- Point-)[F]

    # Cotangent weights assemble edge readings into the direction-independent frame Dirichlet energy density.
    tails, heads = topology.corners[2, 1]
    dots = -sides[tails].dot(sides[heads])                                   # [3] Scalar+[F]
    cotangents = dots / (2 * areas)                                          # [3] Scalar+[F]
    energy_density = ((cotangents / (2 * areas)) * face_rates(sides).scalar_norm_squared()).sum(axis=0)  # Scalar+[F]

    # Commuting the two edge turning rates and pairing with the equatorial patch gives the director's swept area.
    swept_turns = face_rates(sides[1]).commutator(face_rates(sides[0])) / 2  # Bivector+[F]
    solid_angles = swept_turns & mv.xy.dual()                                # Scalar+[F]
    return rotors, directors, scalar_parts, bivector_norms, energy_density, solid_angles


skyrmion_rotors, directors, scalar_parts, bivector_norms, skyrmion_energy, solid_angles = skyrmion_winding(
    skyrmion_topology, skyrmion_vertices, scenarios.skyrmion_twist,
)
skyrmion_charge = solid_angles.sites.sum() / (4 * np.pi)                     # [] Scalar+

save_figure(
    render.skyrmion_figure(skyrmion_topology, skyrmion_vertices, directors),
    "frame_fields_skyrmion",
);

Matched colours trace the same curves on the disk at left and on the unit director sphere at right: radial spokes map from the coral north-pole marker to meridians meeting at the blue south-pole marker, while concentric grey rings map to latitude circles that expand to the equator and contract back to the south pole.

In [ ]:
print("Skyrmion topological charge:", skyrmion_charge.to_array())

## Harmonic boundary extension across a curved saddle

Prescribed boundary frames extend smoothly across a curved surface when the cotangent Laplacian acts on the four-dimensional even subalgebra. Solving the linear Dirichlet system minimizes chord-difference energy across interior edges, and normalizing each vertex value projects the chord solution back onto unit rotors.

In [ ]:
saddle_topology, saddle_vertices, boundary_indices, prescribed_rotors, initial_rotors = scenarios.saddle_surface(
    scenarios.disk_radius, scenarios.n_saddle_divisions, scenarios.saddle_height,
)

In [ ]:
def rotor_laplacian_operators(
    topology: Topology, vertices: Point,
) -> tuple[SparseExtensor, SparseExtensor, Scalar, Scalar, Point]:
    """Mixed-grade rotor coboundary (`(Even+ <- Even+)[E <- V]`), cotangent Laplacian (`(Even+ <- Even+)[V <- V]`), edge cotangent weights (`Scalar+[E]`), triangle areas (`Scalar+[F]`), and oriented triangle sides (`[3] Point-[F]`) follow from the surface geometry."""
    edges = topology.d[0] * vertices                                         # Point-[E]
    sides = topology.boundary[2, 1] * edges                                  # [3] Point-[F]
    patches = (sides[0] & sides[1]) / 2                                      # Patch-[F]
    areas = patches.norm()                                                   # Scalar+[F]
    tails, heads = topology.corners[2, 1]
    dots = -sides[tails].dot(sides[heads])                                   # [3] Scalar+[F]
    cotangents = dots / (2 * areas)                                          # [3] Scalar+[F]
    edge_weights = (topology.incidence[2, 1].adjugate() * (cotangents / 2)).sum(axis=0)  # Scalar+[E]
    rotor_d0 = topology.d[0] * Rotor                                         # (Even+ <- Even+)[E <- V]
    rotor_laplacian = rotor_d0.adjoint()(spdiag(edge_weights) * rotor_d0)    # (Even+ <- Even+)[V <- V]
    return rotor_d0, rotor_laplacian, edge_weights, areas, sides


def harmonic_rotor_frames(
    topology: Topology, vertices: Point, boundary_indices: np.ndarray, prescribed_rotors: Rotor,
) -> tuple[Even, Rotor]:
    """Boundary rotors (`[...] Rotor+[V]`) on `boundary_indices` extend across interior vertices into a harmonic chord field (`[...] Even+[V]`) and its unit-normalized rotor field (`[...] Rotor+[V]`)."""
    _, rotor_laplacian, _, _, _ = rotor_laplacian_operators(topology, vertices)
    free_vertices = topology.interior[0]
    free_inclusion = topology.reading(free_vertices, 0).adjugate() * Rotor   # (Even+ <- Even+)[V <- V_free]
    boundary_inclusion = topology.reading(boundary_indices, 0).adjugate() * Rotor  # (Even+ <- Even+)[V <- V_boundary]

    # Solving in the 4D even subalgebra minimizes chord Dirichlet energy; normalization restores unit rotors.
    boundary_values = boundary_inclusion(boundary_inclusion.adjoint()(prescribed_rotors))  # [...] Even+[V]
    rhs = -free_inclusion.adjoint()(rotor_laplacian(boundary_values))        # [...] Even+[V_free]
    harmonic_chords = boundary_values + free_inclusion(
        free_inclusion.adjoint()(rotor_laplacian(free_inclusion)).solve(rhs)
    )                                                                        # [...] Even+[V]
    harmonic_rotors = harmonic_chords.normalized()                           # [...] Rotor+[V]
    return harmonic_chords, harmonic_rotors


rotor_d0, rotor_laplacian, edge_weights, _, _ = rotor_laplacian_operators(saddle_topology, saddle_vertices)
harmonic_chords, harmonic_rotors = harmonic_rotor_frames(
    saddle_topology, saddle_vertices, boundary_indices, prescribed_rotors,
)

save_figure(
    render.rotor_frames_figure(saddle_topology, saddle_vertices, harmonic_rotors),
    "frame_fields_frames",
);

Concentric ribbons ride fixed rings above the translucent saddle while their coral-to-blue cross-sections and gold/teal rims turn with the local rotor frame, comparing gentle boundary tilt on the left against opposite boundary twists in the middle and right panels.

## Unit-constrained rotor relaxation

When an initial frame field starts with extra interior twist, projected cotangent steps unwind that twist while keeping every vertex on the unit rotor sphere and holding the boundary frames fixed. Enforcing unit norm at every step solves the harmonic-map problem on the rotor sphere rather than normalizing a single chord solve afterwards.

In [ ]:
def rotor_relaxation(
    topology: Topology, vertices: Point, initial_rotors: Rotor,
    step_size: float, n_steps: int,
) -> Iterator[Rotor]:
    """Yields the unit rotor field (`Rotor+[V]`) over `n_steps + 1` projected cotangent smoothing steps with fixed boundary frames."""
    _, rotor_laplacian, _, _, _ = rotor_laplacian_operators(topology, vertices)
    free_vertices = topology.interior[0]
    free_inclusion = topology.reading(free_vertices, 0).adjugate() * Rotor   # (Even+ <- Even+)[V <- V_free]
    current_rotors = initial_rotors                                          # Rotor+[V]
    for _ in range(n_steps + 1):
        yield current_rotors
        current_rotors = (
            current_rotors - step_size * free_inclusion(free_inclusion.adjoint()(rotor_laplacian(current_rotors)))
        ).normalized()                                                       # Rotor+[V]


streamed_rotors = show_states(
    rotor_relaxation(
        saddle_topology, saddle_vertices, initial_rotors,
        scenarios.relaxation_step_size, scenarios.n_relaxation_steps,
    ),
    render.relaxation_animation,
    saddle_topology, saddle_vertices,
    name="frame_fields_relaxation",
    duration_ms=scenarios.frame_duration_ms,
    n_frames=scenarios.n_relaxation_steps + 1,
);

Across the animation, the interior ribbons untwist smoothly toward the boundary-prescribed orientation while the outer ring stays locked to its pinned boundary frames.

In [ ]:
trajectory_rotors = stack(streamed_rotors)                                   # [n_relaxation_steps + 1] Rotor+[V]
relaxation_energies = (edge_weights * rotor_d0(trajectory_rotors).scalar_norm_squared()).sites.sum() / 2  # [n_relaxation_steps + 1] Scalar+
print("Initial vs relaxed Dirichlet energy:", relaxation_energies[[0, -1]].to_array())
print("Normalized chord-field energies across the boundary cases:",
      ((edge_weights * rotor_d0(harmonic_rotors).scalar_norm_squared()).sites.sum() / 2).to_array())

In [ ]:
# checks
probe_step = corners[1] - corners[0]                                         # [] Point-
assert ((darboux(probe_step).select[2] - whitney_rates(probe_step)).scalar_norm_squared() < 1e-24).all()
assert ((darboux(probe_step).select[0] - chord_stretch(probe_step)).abs() < 1e-12).all()
assert ((unit_rotors.scalar_norm_squared() - 1).abs() < 1e-12).all()

sides = (triangle.boundary[2, 1] * (triangle.d[0] * corners.field())).batch()[:, 0]  # [3] Point-
patch = (sides[0] & sides[1]) / 2                                            # [] Patch-
radial_whitney = (((points[:, None] - corners) | patch.inverse()) / 2) * triangle.orientation[1]  # [..., 3] Plane+
assert ((whitney_forms - radial_whitney).scalar_norm_squared() < 1e-24).all()

assert bool((skyrmion_charge - 1).abs() < 1.5e-2)
assert ((scalar_parts.squared() + bivector_norms.squared() - 1).abs() < 1e-12).all()

free_inclusion = saddle_topology.reading(saddle_topology.interior[0], 0).adjugate() * Rotor
assert (free_inclusion.adjoint()(rotor_laplacian(harmonic_chords)).scalar_norm_squared() < 1e-20).all()
assert (relaxation_energies[1:] < relaxation_energies[:-1]).all()